<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Tops__cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%capture
# @title command install
!pip install polars scrapling patchright msgspec browserforge nest_asyncio beautifulsoup4 playwright xlsxwriter -q
!patchright install chromium
!patchright install-deps

In [8]:
import asyncio
import nest_asyncio
import polars as pl
from datetime import date
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
from playwright.async_api import async_playwright
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-28


In [4]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return


In [9]:
# ============================================================
# 1. Environment Setup
# ============================================================

# %%capture
# !pip install polars scrapling patchright msgspec browserforge nest_asyncio beautifulsoup4 -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from datetime import date
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright
from IPython.display import display

nest_asyncio.apply()


# ============================================================
# UDF: Data Prep & Transformation
# ============================================================

def process_tops_data(df: pl.DataFrame) -> pl.DataFrame:

    if df.is_empty():
        return df

    df = (
        df.with_columns([
            pl.col("promotion_price").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.Float64, strict=False)
        ])
        .with_columns(
            pl.when(
                pl.col("original_price").is_null() &
                pl.col("promotion_price").is_not_null()
            )
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            pl.when(
                pl.col("promotion_price") == pl.col("original_price")
            )
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

    today = date.today().strftime("%Y-%m-%d")
    quant_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"

    return df.with_columns([
        pl.lit(today).alias("Date"),
        pl.col("name").str.split(" ").list.first().alias("Brand"),
        pl.col("name").str.extract(quant_pattern, 1)
            .cast(pl.Int64, strict=False).alias("Volume"),
        pl.col("name").str.extract(quant_pattern, 2)
            .str.to_uppercase().alias("Unit"),
        pl.lit("Tops").alias("Retailer")
    ])


# ============================================================
# Single URL Scraper
# ============================================================

async def scrape_tops_full_catalog(url: str) -> pl.DataFrame:

    extracted_data = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )

        await context.add_cookies([
            {'name': 'language', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'},
            {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'}
        ])

        page = await context.new_page()

        print(f"Opening Tops: {url}")

        try:
            await page.goto(url, wait_until="load", timeout=90000)
            await asyncio.sleep(5)
        except Exception as e:
            print(f"Initial load error: {e}")
            await browser.close()
            return pl.DataFrame()

        previous_count = 0
        retries = 0

        for attempt in range(60):

            await page.keyboard.press("PageDown")
            await asyncio.sleep(2)

            await page.evaluate("window.scrollBy(0, 1000)")
            await asyncio.sleep(3)

            current_items = await page.query_selector_all(
                'div[class*="text-textblack"]'
            )
            current_count = len(current_items)

            if current_count > previous_count:
                print(f"Items detected: {current_count}...")
                previous_count = current_count
                retries = 0
            else:
                retries += 1
                await page.evaluate("window.scrollBy(0, -1500)")
                await asyncio.sleep(2)
                await page.keyboard.press("End")
                await asyncio.sleep(2)

            if retries >= 5:
                print("End of catalog or scroll limit reached.")
                break

        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")

        name_tags = soup.find_all(
            "div",
            class_=lambda x: x and "text-textblack" in x
        )

        for nt in name_tags:

            name = nt.get_text(strip=True)

            parent = (
                nt.find_parent(
                    "div",
                    class_=lambda x: x and ("relative" in x or "item" in x)
                )
                or nt.parent
            )

            price_spans = parent.find_all(
                "span",
                class_="hidden lg:inline"
            )

            if len(price_spans) >= 2:
                promo = price_spans[0].get_text(strip=True).replace(',', '')
                orig = price_spans[1].get_text(strip=True).replace(',', '')
            elif len(price_spans) == 1:
                promo = None
                orig = price_spans[0].get_text(strip=True).replace(',', '')
            else:
                promo = orig = None

            extracted_data.append({
                "name": name,
                "promotion_price": promo,
                "original_price": orig
            })

        await browser.close()

    df_raw = pl.DataFrame(extracted_data)
    return process_tops_data(df_raw)


# ============================================================
# Multi URL Scraper ✅ FIXED
# ============================================================

async def scrape_tops_multi_url(urls: list[str]) -> pl.DataFrame:

    temp_df = pl.DataFrame()

    for url in urls:
        try:
            df = await scrape_tops_full_catalog(url)
            temp_df = pl.concat([temp_df, df])
        except Exception as e:
            print(f"Error with {url}: {e}")

    return temp_df

Tops has limited query MAX 40.

In [6]:
%%skip
# --- EXECUTION ---
# Can GET MAX 40 items per URL
urls_to_scrape = [
    "https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=HYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=ATTACK%2CHYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=PAO",
    "https://www.tops.co.th/en/household-and-pet/laundry/concentrated-fabric-softener?brand=HYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/concentrated-fabric-softener?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/powder-detergent?brand=ATTACK%2CPAO%2CPRO",
    "https://www.tops.co.th/en/household-and-pet/laundry/regular-fabric-softener?brand=HYGIENE%2CFINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/gel-ball-detergent?brand=PAO",
    "https://www.tops.co.th/en/household-and-pet/dish-cleaner/dish-detergent?brand=LIPON+F"
]

# # ✅ MUST await async function
# tops_df = await scrape_tops_multi_url(urls_to_scrape)

# # Display
# if not tops_df.is_empty():
#     display(tops_df)
#     print(f"Grand Total Items Found: {len(tops_df)}")
# else:
#     print("No data collected.")

### Watchlist

In [10]:
# @title watchlist queue unlimited
import random
import asyncio
import polars as pl
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright

async def scrape_tops_watchlist_unlimited(urls: list[str]) -> pl.DataFrame:
    """
    Unlimited Queue Scraper.
    Never drops a URL. Will keep pushing failed URLs to the back of the queue
    until every single item is successfully scraped.
    """
    extracted_data = []

    # Initialize the queue as a list of tuples: (url, current_attempt_number)
    queue = [(url, 1) for url in urls]

    print(f"Starting UNLIMITED scrape for {len(urls)} products...")
    print("Strategy: Infinite Queue + Heavy Cooldowns for stubborn links.\n")

    while queue:
        # Pop the first item off the front of the queue
        current_url, attempt = queue.pop(0)

        print(f"Fetching: {current_url}")
        print(f"  -> (Attempt {attempt}) | Items remaining in queue: {len(queue) + 1}")

        success = False

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                viewport={"width": 1920, "height": 1080}
            )

            await context.add_cookies([
                {'name': 'language', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'},
                {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'}
            ])

            page = await context.new_page()

            try:
                await page.goto(current_url, wait_until="domcontentloaded", timeout=60000)

                await page.wait_for_selector("h1", timeout=15000)
                h1_locator = page.locator("h1").first
                h1_text = await h1_locator.inner_text()

                # CLOUDFLARE CHECK
                if "tops.co.th" in h1_text.lower() or "just a moment" in h1_text.lower() or "blocked" in h1_text.lower():
                    print("  -> Cloudflare challenge gate detected. Attempting active bypass...")

                    resolved = False
                    for tick in range(30):
                        await asyncio.sleep(1)
                        await page.mouse.move(random.randint(300, 800), random.randint(300, 800))
                        if tick % 5 == 0:
                            await page.mouse.click(960, 540) # Click Turnstile

                        h1_text = await h1_locator.inner_text()
                        if "tops.co.th" not in h1_text.lower() and "just a moment" not in h1_text.lower() and "blocked" not in h1_text.lower():
                            print("  -> Cloudflare resolved!")
                            resolved = True
                            break

                    if not resolved:
                        print("  -> Still blocked after 30s. Triggering emergency reload...")
                        await page.reload(wait_until="domcontentloaded", timeout=60000)
                        await asyncio.sleep(5)
                        h1_text = await h1_locator.inner_text()

                        if "tops.co.th" not in h1_text.lower() and "just a moment" not in h1_text.lower() and "blocked" not in h1_text.lower():
                            resolved = True
                            print("  -> Cloudflare resolved after reload!")
                        else:
                            print("  -> Bypass failed on this turn.")
                else:
                    resolved = True # No CF challenge

                if resolved:
                    await asyncio.sleep(2) # Let React load the prices
                    html = await page.content()
                    soup = BeautifulSoup(html, "html.parser")

                    name_tag = soup.find("h1")
                    name = name_tag.get_text(strip=True) if name_tag else None

                    price_spans = soup.find_all("span", class_="hidden lg:inline")
                    promo = orig = None
                    if len(price_spans) >= 2:
                        promo = price_spans[0].get_text(strip=True).replace(',', '').replace('฿', '')
                        orig = price_spans[1].get_text(strip=True).replace(',', '').replace('฿', '')
                    elif len(price_spans) == 1:
                        orig = price_spans[0].get_text(strip=True).replace(',', '').replace('฿', '')

                    condition_tag = soup.find("div", class_=lambda c: c and "text-textsecondary" in c and "text-sm" in c and "mt-2" in c)
                    condition = condition_tag.get_text(strip=True) if condition_tag else None

                    if name and "tops.co.th" not in name.lower():
                        extracted_data.append({
                            "name": name,
                            "promotion_price": promo,
                            "original_price": orig,
                            "condition": condition
                        })
                        promo_display = f" | Promo: {condition}" if condition else ""
                        print(f"  Successfully grabbed: {name}{promo_display}")
                        success = True
                    else:
                        print("  -> Extracted name is invalid. Likely still blocked.")

            except Exception as e:
                print(f"  Error: {str(e)[:80]}...")

            finally:
                if browser.is_connected():
                    await browser.close()

        # UNLIMITED QUEUE LOGIC: Always re-queue on failure
        if not success:
            # Push it to the BACK of the queue to try again later
            queue.append((current_url, attempt + 1))
            print(f"  [!] Failed. Pushing to the back of the queue (Will never drop).\n")

            # Dynamic Cooldowns based on remaining queue size
            if len(queue) == 1:
                print("  (Only 1 item left in queue. Taking a 30s heavy cooldown to reset IP flags...)")
                await asyncio.sleep(30)
            elif len(queue) <= 3:
                print("  (Queue is getting small. Taking a 15s cooldown...)")
                await asyncio.sleep(15)
            else:
                await asyncio.sleep(random.uniform(5, 8))
        else:
            # Successful! Just wait a few seconds before processing the next item in the queue
            wait_time = random.uniform(3, 5)
            if queue:
                print(f"  Waiting {wait_time:.1f}s before next item in queue...\n")
            await asyncio.sleep(wait_time)

    if not extracted_data:
        print("\nNo data collected.")
        return pl.DataFrame()

    df_raw = pl.DataFrame(extracted_data)
    return process_tops_data(df_raw)


# --- RUN IT ---
watchlist = [
    "https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-550ml-8851989033365",
    "https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-1250ml-8851989034737",
    "https://www.tops.co.th/en/hygiene-expert-wash-concentrated-liquid-detergent-milky-touch-scent-600ml-8850092254155",
    "https://www.tops.co.th/en/hygiene-expert-wash-concentrated-liquid-detergent-milky-touch-scent-1400ml-8850092254216",
    "https://www.tops.co.th/en/pao-win-wash-liquid-concentrated-detergent-620ml-refill-8850002024823",
    "https://www.tops.co.th/en/pao-win-wash-liquid-concentrated-detergent-1300ml-refill-8850002031739",
    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-480ml-8850092280604",
    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-480ml-pack-3-8850092280819",
    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-1000ml-8850092280901",
    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-1000ml-pack-2-8850092280925",
    "https://www.tops.co.th/en/lipon-f-x-tra-hygienic-dish-wash-500ml-pack-3-8850002042643",
    "https://www.tops.co.th/en/lipon-f-dish-wash-3-2ltr-8850002010772"
]

df_watchlist_results = await scrape_tops_watchlist_unlimited(watchlist)
display(df_watchlist_results)

Starting UNLIMITED scrape for 12 products...
Strategy: Infinite Queue + Heavy Cooldowns for stubborn links.

Fetching: https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-550ml-8851989033365
  -> (Attempt 1) | Items remaining in queue: 12
  Successfully grabbed: Fineline Liquid Detergent Sunny Gold 550ml. | Promo: Buy 2 Pay 1
  Waiting 3.2s before next item in queue...

Fetching: https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-1250ml-8851989034737
  -> (Attempt 1) | Items remaining in queue: 11
  Successfully grabbed: Fineline Liquid Detergent Sunny Gold 1250ml. | Promo: Buy 2 Pay 1
  Waiting 3.6s before next item in queue...

Fetching: https://www.tops.co.th/en/hygiene-expert-wash-concentrated-liquid-detergent-milky-touch-scent-600ml-8850092254155
  -> (Attempt 1) | Items remaining in queue: 10
  Successfully grabbed: Hygiene Expert Wash Concentrated Liquid Detergent Milky Touch Scent 600ml. | Promo: Sale
  Waiting 4.8s before next item in queue...

Fetching:

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Retailer
str,f64,f64,str,str,str,i64,str,str
"""Fineline Liquid Detergent Sunn…",null,89.0,"""Buy 2 Pay 1""","""2026-04-28""","""Fineline""",550,"""ML""","""Tops"""
"""Fineline Liquid Detergent Sunn…",null,179.0,"""Buy 2 Pay 1""","""2026-04-28""","""Fineline""",1250,"""ML""","""Tops"""
"""Hygiene Expert Wash Concentrat…",65.0,55.0,"""Sale""","""2026-04-28""","""Hygiene""",600,"""ML""","""Tops"""
"""Hygiene Expert Care Concentrat…",219.0,195.0,"""Sale""","""2026-04-28""","""Hygiene""",1000,"""ML""","""Tops"""
"""Lipon F X Tra Hygienic Dish Wa…",87.0,65.0,null,"""2026-04-28""","""Lipon""",500,"""ML""","""Tops"""
…,…,…,…,…,…,…,…,…
"""Pao Win Wash Liquid Concentrat…",99.0,96.0,null,"""2026-04-28""","""Pao""",620,"""ML""","""Tops"""
"""Pao Win Wash Liquid Concentrat…",null,185.0,"""Buy 2 Pay 1""","""2026-04-28""","""Pao""",1300,"""ML""","""Tops"""
"""Hygiene Expert Care Concentrat…",129.0,109.0,"""Sale""","""2026-04-28""","""Hygiene""",1000,"""ML""","""Tops"""


In [12]:
# concat watchlist
df_tops = df_watchlist_results.clone()

In [13]:
import polars as pl

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. Swaps original and promotion if original < promotion.
    2. If original_price is missing, fills it with promotion_price.
    3. If promotion_price matches original_price, sets promotion to Null.
    """
    return (
        df.with_columns(
            # Create temporary columns to determine the true Max and Min
            # This automatically handles the "swap" and "missing original" logic
            pl.max_horizontal("original_price", "promotion_price").alias("original_price"),
            pl.min_horizontal("original_price", "promotion_price").alias("temp_promo")
        )
        .with_columns(
            # Finalize promotion_price: if it equals original_price, it's not a promotion
            pl.when(pl.col("temp_promo") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("temp_promo"))
            .alias("promotion_price")
        )
        .drop("temp_promo") # Clean up helper column
    )

# Usage:
df_prep_tops = re_evaluate_price(df_tops)

In [14]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int16, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )



In [15]:
# incorrect result
df_trans_tops = parse_product_names(df_prep_tops, "Tops")

In [16]:
df_trans_tops

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Retailer,Pack
str,f64,f64,str,str,str,i16,str,str,str
"""Fineline Liquid Detergent Sunn…",null,89.0,"""Buy 2 Pay 1""","""2026-04-28""","""Fineline""",550,"""ML""","""Tops""",null
"""Fineline Liquid Detergent Sunn…",null,179.0,"""Buy 2 Pay 1""","""2026-04-28""","""Fineline""",1250,"""ML""","""Tops""",null
"""Hygiene Expert Wash Concentrat…",55.0,65.0,"""Sale""","""2026-04-28""","""Hygiene""",600,"""ML""","""Tops""",null
"""Hygiene Expert Care Concentrat…",195.0,219.0,"""Sale""","""2026-04-28""","""Hygiene""",1000,"""ML""","""Tops""","""PACK 2"""
"""Lipon F X Tra Hygienic Dish Wa…",65.0,87.0,null,"""2026-04-28""","""Lipon""",500,"""ML""","""Tops""","""PACK 3"""
…,…,…,…,…,…,…,…,…,…
"""Pao Win Wash Liquid Concentrat…",96.0,99.0,null,"""2026-04-28""","""Pao""",620,"""ML""","""Tops""",null
"""Pao Win Wash Liquid Concentrat…",null,185.0,"""Buy 2 Pay 1""","""2026-04-28""","""Pao""",1300,"""ML""","""Tops""",null
"""Hygiene Expert Care Concentrat…",109.0,129.0,"""Sale""","""2026-04-28""","""Hygiene""",1000,"""ML""","""Tops""",null


In [ ]:
# df_trans_tops.unique().write_excel(f"tops_{today_date}.xlsx")

In [17]:
df_prep_tops_watchlist = re_evaluate_price(df_watchlist_results)
df_trans_tops_watchlist = parse_product_names(df_prep_tops_watchlist, "Tops")
# saving Excel file
df_trans_tops_watchlist.unique().write_excel(f"tops_watchlists_{today_date}.xlsx")